In [ ]:
import os
import pandas as pd
import numpy as np
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt
import csv
from pathlib import Path

%load_ext autoreload
%autoreload 2

# Prepare the ADME datasets
The file ADME_public_set_3521.csv is taken from this link: https://github.com/molecularinformatics/Computational-ADME/blob/main/ADME_public_set_3521.csv

In [ ]:
df = pd.read_csv("ADME_public_set_3521.csv")
df.head()

In [ ]:
df.count()

In [ ]:
file_name_dict = {
    "LOG HLM_CLint (mL/min/kg)": "HLM",
    "LOG MDR1-MDCK ER (B-A/A-B)": "MDR1_MDCK_ER",
    "LOG SOLUBILITY PH 6.8 (ug/mL)": "Solubility",
    "LOG RLM_CLint (mL/min/kg)": "RLM",
    "LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)": "hPPB",
    "LOG PLASMA PROTEIN BINDING (RAT) (% unbound)": "rPPB",
}
for key in file_name_dict.keys():
    sub_df = df[["SMILES", key]]
    sub_df = sub_df.dropna()
    sub_df.columns = ["smiles", "activity"]
    print(f"{key}: {len(sub_df)}")
    file_name = file_name_dict[key]
    sub_df.to_csv(f"{file_name}.csv", index=False)

# Split QM9 dataset into different sets based on composition or target value

In [ ]:
df = pd.read_csv("qm9.csv")
df

## Heavy atoms

In [ ]:
def count_heavy_atoms(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol.GetNumHeavyAtoms()


# Apply the function to the dataframe
df["num_heavy_atoms"] = df["smiles"].apply(count_heavy_atoms)
df["num_heavy_atoms"].value_counts()

In [ ]:
train_df = df[df["num_heavy_atoms"].isin([7, 8, 9])]
train_indices = sorted(train_df.sample(n=100000, random_state=42).index)
test_indices = sorted(list(set(range(len(df))) - set(train_indices)))
train_df = df.loc[train_indices]
test_df = df.loc[test_indices]

parent = Path("qm9/heavy_atoms_h298")
parent.mkdir(exist_ok=True)

train_df[["smiles", "h298"]].to_csv("qm9/heavy_atoms_h298/train.csv", index=False)
test_df[["smiles", "h298"]].to_csv("qm9/heavy_atoms_h298/test.csv", index=False)

## h298 value splits (based on -350 kcal/mol)

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df["h298"], bins=30, edgecolor="black")
plt.title("Distribution of Values")
plt.xlabel("Values")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

train_df = df[df["h298"] < -350]
train_indices = sorted(train_df.sample(n=100000, random_state=42).index)
test_indices = sorted(list(set(range(len(df))) - set(train_indices)))
train_df = df.loc[train_indices]
test_df = df.loc[test_indices]

parent = Path("qm9/values_split_h298")
parent.mkdir(exist_ok=True)

train_df[["smiles", "h298"]].to_csv("qm9/values_split_h298/train.csv", index=False)
test_df[["smiles", "h298"]].to_csv("qm9/values_split_h298/test.csv", index=False)

# QM9 noisy dataset split / nitorgen ingoring dataset split

## QM9 systemamtically error (homoscedastic)

In [ ]:
noise = 5
noise_dir = os.path.join("qm9", f"noise{noise}")
if not os.path.exists(noise_dir):
    os.mkdir(noise_dir)
with open("qm9.csv", "r") as f:
    reader = csv.reader(f)
    header = next(reader)
    lines = []
    for line in reader:
        lines.append([line[0], float(line[-2]) + np.random.normal(scale=noise)])

header = [header[0], header[-2]]
new_file = f"qm9_noise{noise}.csv"
with open(os.path.join(noise_dir, new_file), "w") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(lines)

## QM9 systemamtically error (heteroscedastic)

In [ ]:
noise = 5
noise_dir = os.path.join("qm9", f"noise{noise}_N")
if not os.path.exists(noise_dir):
    os.mkdir(noise_dir)
with open("qm9.csv", "r") as f:
    reader = csv.reader(f)
    header = next(reader)
    lines = []
    for line in reader:
        if "N" in line[0] or "n" in line[0]:
            lines.append([line[0], float(line[-2]) + np.random.normal(scale=noise)])
        else:
            lines.append([line[0], float(line[-2])])

header = [header[0], header[-2]]
new_file = f"qm9_noise{noise}_N.csv"
with open(os.path.join(noise_dir, new_file), "w") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(lines)

## N-ignoring datset (domain shift)

In [ ]:
train_indices = sorted(train_df.sample(n=100000, random_state=42).index)
test_indices = sorted(list(set(range(len(df))) - set(train_indices)))
train_df = df.loc[train_indices]
test_df = df.loc[test_indices]

In [ ]:
df = pd.read_csv("qm9.csv")
df["has_N"] = df.iloc[:, 0].apply(lambda x: "N" in x or "n" in x)
non_N_indices = df[df["has_N"] == False].sample(n=40000, random_state=42).index
train_indices = sorted(non_N_indices.tolist())
test_indices = sorted(list(set(range(len(df))) - set(train_indices)))

In [ ]:
noise_dir = os.path.join("qm9", "N_split")
if not os.path.exists(noise_dir):
    os.mkdir(noise_dir)

df = pd.read_csv("qm9.csv")
df["has_N"] = df.iloc[:, 0].apply(lambda x: "N" in x or "n" in x)
non_N_indices = df[df["has_N"] == False].sample(n=40000, random_state=42).index
train_indices = sorted(non_N_indices.tolist())
test_indices = sorted(list(set(range(len(df))) - set(train_indices)))

train_df = df.loc[train_indices]
test_df = df.loc[test_indices]
train_df[["smiles", "h298"]].to_csv(f"{noise_dir}/train.csv", index=False)
test_df[["smiles", "h298"]].to_csv(f"{noise_dir}/test.csv", index=False)
print(len(train_df))
print(len(test_df))

# Split Stokes's dataset into 10 folds

In [ ]:
df = pd.read_csv("stokes_primary_regr.csv")
df

In [ ]:
for i in range(10):
    save_dir = f"stokes_primary_regr/fold_{i}"
    os.makedirs(save_dir, exist_ok=True)
    df = pd.read_csv("stokes_primary_regr.csv")
    train_size = int(0.8 * len(df))
    train_val_size = len(df)
    df = df.sample(frac=1, random_state=i).reset_index(drop=True)
    train = df[:train_size]
    val = df[train_size:train_val_size]

    # Sample indices for the training set
    train_high_inhibition_indices = [
        j for j, v in enumerate(train["Mean_Inhibition"].tolist()) if v > 0.2
    ]
    num_train_high_inhibition_samples = int(0.1 * len(train_high_inhibition_indices))
    sampled_train_high_inhibition_indices = np.random.choice(
        train_high_inhibition_indices, num_train_high_inhibition_samples, replace=False
    )

    train_indices = [
        j for j, v in enumerate(train["Mean_Inhibition"].tolist()) if v <= 0.2
    ]
    train_indices.extend(sampled_train_high_inhibition_indices)

    # Sample indices for the validation set
    val_high_inhibition_indices = [
        j for j, v in enumerate(val["Mean_Inhibition"].tolist()) if v > 0.2
    ]
    num_val_high_inhibition_samples = int(0.1 * len(val_high_inhibition_indices))
    sampled_val_high_inhibition_indices = np.random.choice(
        val_high_inhibition_indices, num_val_high_inhibition_samples, replace=False
    )

    val_indices = [j for j, v in enumerate(val["Mean_Inhibition"].tolist()) if v <= 0.2]
    val_indices.extend(sampled_val_high_inhibition_indices)

    train = train.iloc[train_indices]
    val = val.iloc[val_indices]

    train.to_csv(os.path.join(save_dir, "train.csv"), index=False)
    val.to_csv(os.path.join(save_dir, "val.csv"), index=False)

    print(f"Fold {i}: {len(train)} training samples, {len(val)} validation samples")